# Tesi QML — Colab runner

Orchestratore sottile per la matrice PennyLane: prepara la VM, richiama gli script testabili e salva atomicamente gli artefatti su Drive. Modelli, training, metriche e statistica restano nel package.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess
import torch

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/tesi_qml_colab')
PROJECT_ROOT = Path('/content/tesi_finale')
CODE_ARCHIVE = DRIVE_ROOT / 'colab_code.tar.gz'
DATASET_ARCHIVE = DRIVE_ROOT / 'colab_dataset.tar.gz'
STATE_ARCHIVE = DRIVE_ROOT / 'colab_state.tar.gz'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def run(command):
    print('$', ' '.join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True)

print({'device': DEVICE, 'project': str(PROJECT_ROOT)})

## Preparazione e verifica

In [ ]:
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)
run(['tar', '-xzf', CODE_ARCHIVE, '-C', PROJECT_ROOT])
run(['tar', '-xzf', DATASET_ARCHIVE, '-C', PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-cluster.txt'])
run(['python', '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
import torch
run(['python', 'scripts/check_storage.py', '--project-root', PROJECT_ROOT])
run(['python', '-m', 'pytest', '-m', 'not quantum', '-q'])
run(['python', '-m', 'pytest', 'tests/quantum/test_pennylane_qnn.py', '-q'])

## Ripristino e sessione riprendibile

In [ ]:
SESSION_RUN_LIMIT = 4
run([
    'python', 'scripts/colab_session.py', '--project-root', PROJECT_ROOT,
    '--state-archive', STATE_ARCHIVE, 'run',
    '--max-runs', SESSION_RUN_LIMIT, '--device', DEVICE,
])
run(['python', 'scripts/colab_session.py', '--project-root', PROJECT_ROOT, '--state-archive', STATE_ARCHIVE, 'status'])

## Aggregazione, figure e salvataggio atomico

In [ ]:
matrix_path = PROJECT_ROOT / 'results/matrix/primary.json'
state = json.loads(matrix_path.read_text())
complete = all(row['status'] == 'complete' for row in state['runs'])
if any(row['status'] == 'complete' for row in state['runs']):
    aggregate_command = ['python', 'scripts/aggregate_results.py', '--project-root', PROJECT_ROOT]
    if not complete:
        aggregate_command.append('--allow-incomplete')
    run(aggregate_command)
    if complete:
        run(['python', 'scripts/generate_figures.py', '--project-root', PROJECT_ROOT])

run(['python', 'scripts/colab_session.py', '--project-root', PROJECT_ROOT, '--state-archive', STATE_ARCHIVE, 'save'])
print({'saved': str(STATE_ARCHIVE), 'matrix_complete': complete})